In [ ]:
import requests
from bs4 import BeautifulSoup
GAPELink = "https://www.sjsu.edu/gape/graduate-student-guide/index.php"

In [ ]:
def scrape_gape_info():
    links = []
    content = ""
    response = requests.get(GAPELink)
    if response.status_code != 200:
        raise Exception("Failed to load page")

    soup = BeautifulSoup(response.content, 'html.parser')
    # Find the main element with id = "sjsu-maincontent"
    main_content = soup.find('main', id='sjsu-maincontent')
    if not main_content:
        raise Exception("Main content not found")
    # Extract all elements in main_content. If there is an <a> tag, save text and link to links list and text to content.
    # Other elements, just save the text to content.
    for element in main_content.descendants:
        if element.name == 'a':
            # if link has a .pdf at the end, ignore link
            if element['href'].endswith('.pdf'):
                continue
            links.append({
                'text': element.get_text(strip=True),
                'link': "https://www.sjsu.edu" + element['href']
            })
            content += element.get_text(strip=True) + "\n"
        elif element.name in ['p', 'h1', 'h2', 'h3', 'h4']:
            content += element.get_text(strip=True) + "\n"
    return content, links

In [ ]:
def scrape_GAPE_sub_info(links):
    for item in links:
        print(f"Scraping GAPE Subpage: {item['text']}")
        # if the link is a pdf, ignore
        if item['link'].endswith('.pdf'):
            continue
        response = requests.get(item['link'])
        if response.status_code != 200:
            print(f"Failed to load page for {item['text']}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        # Find the main element with id = "sjsu-maincontent"
        main_content = soup.find('main', id='sjsu-maincontent')
        if not main_content:
            print(f"Main content not found for {item['text']}")
            continue
        # Extract text from main_content
        content_text = ""
        for element in main_content.descendants:
            if element.name in ['h1', 'h2', 'h3', 'h4']:
                level = int(element.name[1])
                content_text += '\n' + ('#' * level) + \
                    ' ' + element.get_text(strip=True) + '\n'
            elif element.name == 'p':
                content_text += element.get_text(strip=True) + '\n'
        item['content'] = content_text
    return links

In [ ]:
content, links = scrape_gape_info()
print("\nGAPE Links:")
for link in links:
  print(f"{link['text']}: {link['link']}")


GAPE Links:
Academic Planning: https://www.sjsu.edu/gape/graduate-student-guide/academic-planning/index.php
Advancement to Candidacy: https://www.sjsu.edu/gape/graduate-student-guide/advancement-to-candidacy/index.php
Culminating Experience: https://www.sjsu.edu/gape/graduate-student-guide/culminating-experience/index.php
Graduation: https://www.sjsu.edu/gape/graduate-student-guide/graduation/index.php


In [ ]:
sub_info = scrape_GAPE_sub_info(links)

Scraping GAPE Subpage: Academic Planning
Scraping GAPE Subpage: Advancement to Candidacy
Scraping GAPE Subpage: Culminating Experience
Scraping GAPE Subpage: Graduation


In [ ]:
sub_info[0]

{'text': 'Academic Planning',
 'link': 'https://www.sjsu.edu/gape/graduate-student-guide/academic-planning/index.php',
 'content': '\n# Academic Planning\nAcademic planning is essential as you embark on your graduate career.\xa0Select\xa0an item\xa0below\n                  and identify helpful actions for the beginning of your program.\nAdvising is a vital component in your graduate career and can significantly enhance\n                              your university experience. Each program department is designed with faculty members\n                              who serve as advisors. Graduate Program Coordinator are the primary guides for successful\n                              completion of your degree program.\nGraduate students should seek program department advising early in their graduate\n                              careers and regularly connect with Graduate Program Coordinators throughout their\n                              study. Visit ourConnect Pageto identify your Gr

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Change project path [Anyone who is running this change to project folder]
project_path = '/content/drive/MyDrive/Colab Notebooks/CMPE259FinalProject'

%cd {project_path}

In [ ]:
# Save sub_info locally in the drive
import json

file_path = 'CMPE259FinalContent/gape_info.json'

with open(file_path, 'w') as file:
    json.dump(sub_info, file)